# v26 検証 — heuristic(r0) × tabular(pf1/2/3 GBM, OOF) の n1:n2 アンサンブル

**問い**: v25 の heuristic モデル(smooth PF **r0** + beam + hold, LB6.786と同一エンジン)と、
v26 の tabular GBM(特徴の PF は **pf1/2/3** smooth PF)を **n1:n2 で重み付き平均**したとき overall RMSE は改善するか？

**validation の揃え方(重要)**:
- tabular は **OOF**(GroupKFold(5) by well, 学習済み fold pkl を再ロードして各 va fold を予測)で作る → **リーク無し**。
- heuristic は真値を学習に使わない(r0固定 + カテゴリ定数 beam/hold)ので、train 全坑井に適用しても OOF と比較可能。
  ※heuristic のカテゴリ閾値/beam/hold は train で決めた低自由度定数(=6.786を出した実物)なので、train 上の値を fair な代理とする。
- 両者とも **各 id(=`{wid}_{行index}`)で突合**。heuristic の `ev_ids` は parquet の `id` と完全一致。

**指標**: 全坑井の eval 行を1つにプールした **overall RMSE**(=Kaggle指標)。true = `last_known_tvt + target`。

**注意**: tabular の後処理は **submit 準拠(pf_res = pf_ancc − last_known で残差化)**。
train_models の OOF セルは pf_ancc を絶対のまま blend している次元不整合があるため、こちらでは正しい submit 版を使う。

**実行**: このノートは学習済み `artifacts_v26/models`(fold pkl 9×5) と `train.parquet`、コンペ train/ を要する。
heuristic の smooth PF(r0, lag192, n_seed32)を train 全坑井に回すので GPU 推奨(CHUNK 単位で進捗表示)。


In [1]:
# ===== import + パス自動判定(コンペ train / v26 artifacts) =====
import os, glob, json, warnings, time
import numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.signal import savgol_filter
from numba import njit
from sklearn.model_selection import GroupKFold
import joblib
warnings.filterwarnings("ignore"); np.seterr(all="ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; DTYPE = torch.float32

CANDS = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction",
]
COMP = next((p for p in CANDS if os.path.exists(os.path.join(p, "sample_submission.csv"))), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else CANDS[-1]
COMP = Path(COMP); TRAIN = COMP / "train"

CANDS_ART = [Path("/kaggle/input/rogii-v26-artifacts"),
             Path(r"C:\Users\kosaka256\Documents\rogii_claude\v26\artifacts_v26")]
ART = next((p for p in CANDS_ART if (p / "models").exists()), CANDS_ART[-1])
PARQUET = ART / "data" / "train.parquet"
print("device:", DEVICE, "| COMP:", COMP, "| train:", TRAIN.exists())
print("ART:", ART, "| parquet:", PARQUET.exists(), "| models:", (ART / "models").exists())

device: cuda | COMP: C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction | train: True
ART: C:\Users\kosaka256\Documents\rogii_claude\v26\artifacts_v26 | parquet: True | models: True


In [2]:
# ===== 設定 =====
# --- heuristic 側(v25 submit_smoother_v25 と同一。PF=r0固定) ---
SEED         = 0
N_SEED       = 128          # r0_seed32 regime
SMOOTH_LAG   = 512         # 固定ラグ L(v25 best。train -0.229)
SMOOTH_BLEND = 1.0
CHUNK        = 20          # GPUバッチ坑井数。OOMなら下げる
CLAMP_FT     = 60.0
HEUR_MAX_WELLS = None       # None=全坑井。動作確認だけなら小さい数に(例 40)

USE_BEAM = True; USE_HOLD = True
N_EVAL_THRESHOLD  = 4840.0
Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
BEAM_W = {0: 0.0, 1: 0.0, 2: 0.2, 3: 0.0, 4: 0.05, 5: 0.2}
HOLD_W = {0: 0.2, 1: 0.15, 2: 0.15, 3: 0.15, 4: 0.05, 5: 0.05}
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1), (10, 14.0, 90.0, 5),
    (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3), (15, 25.0, 180.0, 2), (20, 30.0, 200.0, 2),
    (15, 10.0, 80.0, 4), (25, 6.0, 50.0, 3), (10, 40.0, 300.0, 1), (12, 18.0, 120.0, 5),
    (30, 8.0, 70.0, 2), (10, 50.0, 400.0, 0),
]
APPLY_PP   = True          # heuristic 側 tauランプ+sg
PP_PARAMS  = {"alpha": 1.0, "tau": 85}
SG_W, SG_P = 17, 3

R0 = {
    "n_particles": 400, "likelihood_scale": 10.536319667317654,
    "init_pos_std": 0.9331541486020234, "init_rate_std": 0.00047753793163756304,
    "mom": 0.9997911117425017, "vn": 0.0006297672017716754, "pn": 0.005014009140727263,
    "rp": 0.7627309890952053, "rr": 0.00045663430474523125, "resamp": 0.032049849143995945,
    "gr_sig_min": 11.865793233968713, "gr_sig_max": 44.98927964612316, "gr_sig_def": 119.25397464636981,
    "gr_sig_mult": 3.058318309943017, "tvt_clip_margin": 150.56111180234504, "rate_clip": 2.0,
    "gr_power": 1.7826896851395453, "hgr_smooth_r": 0, "tw_gr_smooth_r": 0,
    "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 0.013666550145707291, "use_phys": False,
}

# --- tabular(GBM) 後処理 = submit 準拠 ---
PPg = json.loads((ART / "pp_params.json").read_text()) if (ART / "pp_params.json").exists() else {"alpha":1.0,"tau":85,"w_pf":0.09}
USE_SG_GBM = True          # GBM 側も per-well sg(heuristicと揃える)

# --- アンサンブル掃引(heuristic の重み w。tvt = w*heur + (1-w)*gbm) ---
BLEND_GRID = [round(x, 2) for x in np.linspace(0.0, 1.0, 21)]
print(f"heur: PF=r0_seed32 n_seed={N_SEED} lag={SMOOTH_LAG} beam={USE_BEAM} hold={USE_HOLD} pp={APPLY_PP}")
print(f"gbm : PP={PPg} sg={USE_SG_GBM} | blend grid {len(BLEND_GRID)} pts")

heur: PF=r0_seed32 n_seed=128 lag=512 beam=True hold=True pp=True
gbm : PP={'alpha': 1.0, 'tau': 85, 'w_pf': 0.09} sg=True | blend grid 21 pts


In [3]:
# ===== heuristic エンジン(v25 submit_smoother_v25 と同一・インライン) =====
def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def well_code(c):
    n_bin = int(c["n_eval"] > N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(Z_SPAN_THRESHOLDS, c["z_span"], side="right"))
    return n_bin + 2*z_bin

def build_heur(wid):   # v25 build_test を TRAIN 用に(真値TVTは使わない。eval行=TVT_input NaN)
    hw = pd.read_csv(TRAIN / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TRAIN / f"{wid}__typewell.csv").sort_values("TVT")
    if "TVT_input" not in hw.columns or "TVT" not in tw.columns:
        return None
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0 or len(kn) < 1:
        return None
    tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    if len(tw_tvt) < 3:
        return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_filled = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    ev_pos = hw.index.get_indexer(ev.index); kpos = hw.index.get_indexer(kn.index)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    z_ev = ev["Z"].to_numpy(float)
    n_eval = int(len(z_ev)); z_span = float(np.nanmax(z_ev)-np.nanmin(z_ev)) if len(z_ev) else 0.0
    ev_ids = [f"{wid}_{i}" for i in ev.index]
    return dict(md=ev["MD"].to_numpy(float), z=z_ev, gr_filled=gr_filled, ev_pos=ev_pos, kpos=kpos,
                kn_tvtin=kn["TVT_input"].to_numpy(float), tw_tvt=tw_tvt, tw_gr=tw_gr, tw_fb=tw_fb,
                ls=ls, ir=ir, ev_ids=ev_ids, n_eval=n_eval, z_span=z_span,
                last_known_md=float(kn["MD"].iloc[-1]), last_known_tvt=float(kn["TVT_input"].iloc[-1]))

def make_inputs(c, P):
    tw_gr_pf = _smooth(c["tw_gr"], c["tw_fb"], P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(c["tw_tvt"], tw_gr_pf)
    gr_sm = _smooth(c["gr_filled"], c["tw_fb"], P["hgr_smooth_r"])
    if len(c["kpos"]) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[c["kpos"]] - np.interp(c["kn_tvtin"], c["tw_tvt"], tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs*P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"]+1e-6)))
    return dict(md=c["md"], z=c["z"], gr=gr_sm[c["ev_pos"]].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=c["ls"], ir=c["ir"], N=int(P["n_particles"]))

def _pf_smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS*rn(); rate = ir[:, None] + IR*rn()
    w = torch.full((B, N), 1.0/N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype)-1)*step + CLIP
    glast = torch.gather(grid, 1, (glen-1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur-pm).clamp_min(1.0)
        rate_n = ALPHA*rate + RN*rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n*dm[:, None] + PN*rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD*rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n-zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt+zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos-zi; ii = (v-vmin[:, None])/step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen-1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen-2).clamp_min(0)[:, None])
        t = ii-i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c+1)
        eg = gA*(1-t)+gB*t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None]-eg)/gs[:, None]).abs(); dp = d**GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5*dp), zero).clamp_min(1e-300)
        avg = (w*lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate-dipp)/PHYS_SIG; lkp = torch.exp(-0.5*dphys*dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w*lk*lkp, w*lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new/ws.clamp_min(1e-300), torch.full_like(w, 1.0/N))
        ne = (w*w).sum(1); need = act & ((1.0/ne) < (RESAMP*N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype)*(1.0/N)
            u = u0 + arN[None, :]/N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N-1)
            pos_rs = torch.gather(pos, 1, idx) + RP*rn(); rate_rs = torch.gather(rate, 1, idx) + RR*rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0/N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w*(pos-zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i-L] = (w*old).sum(1) - z[:, i-L]
            buf[:, ptr, :] = pos; ptr = (ptr+1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T-L), T):
            pts_s[:, j] = (w*buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik

def _pad(inps, device, dtype=DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn-1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G-1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_smoother_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pf, ps, ll = _pf_smoother_core(
        rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
        pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
        pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
        P, N, device, gen)
    pf = pf.view(W, S, -1).double().cpu().numpy(); ps = ps.view(W, S, -1).double().cpu().numpy()
    ll = ll.view(W, S).cpu().numpy()
    out_f = []; out_s = []
    for w in range(W):
        T = len(inps[w]["md"]); llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)])
            lk = np.where(np.isfinite(llw), llw-mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6)); s = ww.sum()
            ww = ww/s if s > 1e-300 else np.full(S, 1.0/S)
        else:
            ww = np.full(S, 1.0/S)
        out_f.append((ww[:, None]*pf[w, :, :T]).sum(0))
        out_s.append((ww[:, None]*ps[w, :, :T]).sum(0))
    if device == "cuda": torch.cuda.synchronize()
    return out_f, out_s

@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS*6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni])**2/es + mc*(d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i+1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n-1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

def _nn(arr, val): return int(np.abs(arr - val).argmin())

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, int(bs), float(mc), float(es))
    return tw_tvt[path].astype(np.float32)

def run_beam(c):
    tw_tvt = c["tw_tvt"].astype(np.float64)
    fb = float(np.nanmean(c["tw_gr"])) if np.isfinite(np.nanmean(c["tw_gr"])) else 0.0
    tw_gr = np.where(np.isfinite(c["tw_gr"]), c["tw_gr"], fb).astype(np.float64)
    hgr = c["gr_filled"][c["ev_pos"]]; last_tvt = c["last_known_tvt"]
    res = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r) in BEAM_CONFIGS]
    return np.stack(res, 0).mean(0).astype(np.float64)

def post_process(pred, c):
    pred = np.asarray(pred, dtype=np.float64)
    if not APPLY_PP:
        return pred
    md_since = np.maximum(c["md"] - c["last_known_md"], 0.0)
    d = (pred - c["last_known_tvt"]) * (1.0 - np.exp(-md_since / float(PP_PARAMS["tau"]))) * float(PP_PARAMS.get("alpha", 1.0))
    out = c["last_known_tvt"] + d
    n = len(out); wl = min(SG_W, n)
    if wl % 2 == 0: wl -= 1
    return savgol_filter(out, wl, SG_P) if wl >= SG_P + 2 else out
print("heuristic engine ready.")

heuristic engine ready.


In [4]:
# ===== heuristic を TRAIN 全坑井に適用 → id→tvt(絶対) =====
# 対象坑井は parquet と揃える(=学習に使った坑井集合)
meta = pd.read_parquet(PARQUET, columns=["well"])
wells = list(pd.unique(meta["well"])); del meta
if HEUR_MAX_WELLS is not None: wells = wells[:HEUR_MAX_WELLS]
print("target wells:", len(wells))

CACHE = {}
for wid in wells:
    try:
        c = build_heur(str(wid))
    except Exception:
        c = None
    if c is not None: CACHE[str(wid)] = c
usable = [str(w) for w in wells if str(w) in CACHE]
print("usable wells:", len(usable))

P = {**R0, "smooth_lag": SMOOTH_LAG}
heur_pred = {}; t0 = time.time()
for c0 in range(0, len(usable), CHUNK):
    sub = usable[c0:c0+CHUNK]
    inps = [make_inputs(CACHE[w], P) for w in sub]
    out_f, out_s = run_smoother_ens(inps, P, DEVICE, SEED, N_SEED)
    for w, fwd, sm in zip(sub, out_f, out_s):
        c = CACHE[w]
        if CLAMP_FT is not None:
            bad = np.abs(sm - fwd) > CLAMP_FT
            if bad.any(): sm = np.where(bad, fwd, sm)
        pred = (1.0 - SMOOTH_BLEND) * fwd + SMOOTH_BLEND * sm
        code = well_code(c)
        bw = BEAM_W.get(code, 0.0) if USE_BEAM else 0.0
        hwt = HOLD_W.get(code, 0.0) if USE_HOLD else 0.0
        if bw > 0:
            beam = run_beam(c); m = min(len(pred), len(beam))
            pred = np.asarray(pred, dtype=np.float64).copy(); pred[:m] = (1.0 - bw) * pred[:m] + bw * beam[:m]
        if hwt > 0:
            pred = (1.0 - hwt) * pred + hwt * c["last_known_tvt"]
        pred = post_process(pred, c)
        ids = c["ev_ids"]; n = min(len(ids), len(pred))
        for i in range(n): heur_pred[ids[i]] = float(pred[i])
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"  {min(c0+CHUNK,len(usable))}/{len(usable)} ({time.time()-t0:.0f}s)")
print("heuristic ids:", len(heur_pred), f"({time.time()-t0:.0f}s)")

target wells: 773
usable wells: 773
  20/773 (116s)
  40/773 (233s)
  60/773 (359s)
  80/773 (483s)
  100/773 (604s)
  120/773 (719s)
  140/773 (850s)
  160/773 (1000s)
  180/773 (1150s)
  200/773 (1271s)
  220/773 (1402s)
  240/773 (1525s)
  260/773 (1661s)
  280/773 (1776s)
  300/773 (1892s)
  320/773 (2056s)
  340/773 (2198s)
  360/773 (2305s)
  380/773 (2428s)
  400/773 (2564s)
  420/773 (2685s)
  440/773 (2824s)
  460/773 (2956s)
  480/773 (3074s)
  500/773 (3183s)
  520/773 (3289s)
  540/773 (3429s)
  560/773 (3580s)
  580/773 (3702s)
  600/773 (3828s)
  620/773 (3946s)
  640/773 (4065s)
  660/773 (4212s)
  680/773 (4321s)
  700/773 (4471s)
  720/773 (4584s)
  740/773 (4692s)
  760/773 (4822s)
  773/773 (4888s)
heuristic ids: 3783989 (4888s)


In [5]:
# ===== tabular GBM の OOF(学習済み fold pkl を再ロード)→ submit準拠 apply_pp+sg =====
features = json.loads((ART / "features.json").read_text())
df = pd.read_parquet(PARQUET)
ids_arr = df["id"].to_numpy()
y   = df["target"].to_numpy(np.float32)
g   = df["well"].to_numpy()
lk  = df["last_known_tvt"].to_numpy(np.float64)
mds = df["md_since"].to_numpy(np.float64)
pf_abs = df["pf_ancc"].to_numpy(np.float64) if "pf_ancc" in df.columns else lk.copy()  # 絶対TVT
X = df[features].astype("float32"); del df
print("rows", len(X), "| features", len(features), "| wells", pd.unique(g).size)

# --- GroupKFold(5) は無シャッフル=決定的。train_models と同一 folds を再現 ---
folds = list(GroupKFold(n_splits=5).split(X, y, groups=g))
model_dirs = sorted([Path(p) for p in glob.glob(str(ART / "models" / "*")) if os.path.isdir(p)])
model_dirs = [d for d in model_dirs if glob.glob(str(d / "fold*.pkl"))]
print("models:", [d.name for d in model_dirs])

oof_sum = np.zeros(len(y), np.float64); nmod = 0
for d in model_dirs:
    oof = np.zeros(len(y), np.float64); ok = True
    for k, (tr, va) in enumerate(folds):
        fp = d / f"fold{k}.pkl"
        if not fp.exists(): ok = False; break
        oof[va] = joblib.load(fp).predict(X.iloc[va])
    if ok:
        oof_sum += oof; nmod += 1
        from sklearn.metrics import root_mean_squared_error
        print(f"  {d.name}: OOF resid rmse={root_mean_squared_error(y, oof):.4f}")
oof_ens = oof_sum / max(nmod, 1)
print("ensembled models:", nmod)

# --- submit 準拠 apply_pp: pf_res = pf_ancc - lk(残差化), ランプ, alpha ---
w_pf = float(PPg.get("w_pf", 0.09)); tau = PPg.get("tau", 85); alpha = float(PPg.get("alpha", 1.0))
pf_res = pf_abs - lk
d_gbm = oof_ens * (1.0 - w_pf) + pf_res * w_pf
if tau: d_gbm = d_gbm * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
d_gbm = d_gbm * alpha
tvt_gbm = lk + d_gbm

# --- per-well sg(parquet は坑井ごと連続=MD順。heuristic と揃える) ---
if USE_SG_GBM:
    for w in pd.unique(g):
        idx = np.where(g == w)[0]; v = tvt_gbm[idx]; n = len(v); wl = min(SG_W, n)
        if wl % 2 == 0: wl -= 1
        if wl >= SG_P + 2: tvt_gbm[idx] = savgol_filter(v, wl, SG_P)
true_abs = lk + y.astype(np.float64)
gbm_map = {ids_arr[i]: tvt_gbm[i] for i in range(len(ids_arr))}
true_map = {ids_arr[i]: true_abs[i] for i in range(len(ids_arr))}
print("gbm ids:", len(gbm_map))

rows 3783989 | features 511 | wells 773
models: ['catboost-1', 'catboost-2', 'catboost-3', 'lightgbm-1', 'lightgbm-2', 'lightgbm-3', 'xgboost-1', 'xgboost-2', 'xgboost-3']
  catboost-1: OOF resid rmse=7.9994
  catboost-2: OOF resid rmse=7.9865
  catboost-3: OOF resid rmse=8.0493
  lightgbm-1: OOF resid rmse=7.9415
  lightgbm-2: OOF resid rmse=7.8099
  lightgbm-3: OOF resid rmse=7.8211
  xgboost-1: OOF resid rmse=7.9106
  xgboost-2: OOF resid rmse=7.9929
  xgboost-3: OOF resid rmse=8.0569
ensembled models: 9
gbm ids: 3783989


In [6]:
# ===== 突合 → n1:n2 掃引 → overall RMSE =====
def overall_rmse(pred, true):
    e = pred - true; e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e*e)))

common = [i for i in ids_arr if i in heur_pred and i in gbm_map]
print(f"共通 id: {len(common)} / gbm {len(gbm_map)} / heur {len(heur_pred)}")
h = np.array([heur_pred[i] for i in common], np.float64)
tg = np.array([gbm_map[i] for i in common], np.float64)
tr = np.array([true_map[i] for i in common], np.float64)

rmse_h = overall_rmse(h, tr); rmse_g = overall_rmse(tg, tr)
print("\n=== 単体 overall RMSE(共通id, プール=Kaggle指標) ===")
print(f"  heuristic(r0 smooth+beam+hold) : {rmse_h:.4f}")
print(f"  tabular  (pf1/2/3 GBM OOF)     : {rmse_g:.4f}")

print("\n=== ブレンド掃引  tvt = w*heur + (1-w)*gbm ===")
best = (None, 1e9)
for w in BLEND_GRID:
    r = overall_rmse(w*h + (1.0-w)*tg, tr)
    n1 = int(round(w*10)); n2 = 10-n1
    star = ""
    if r < best[1]: best = (w, r)
    print(f"  w_heur={w:4.2f} (heur:gbm≈{n1}:{n2})  rmse={r:.4f}")
print(f"\n★ best: w_heur={best[0]:.2f}  rmse={best[1]:.4f}  "
      f"(heur単体 {rmse_h:.4f} / gbm単体 {rmse_g:.4f} / 改善 {min(rmse_h,rmse_g)-best[1]:+.4f})")
print("※ overall は ±0.2 の seed 床込みで解釈。LB で最終確認。")

共通 id: 3783989 / gbm 3783989 / heur 3783989

=== 単体 overall RMSE(共通id, プール=Kaggle指標) ===
  heuristic(r0 smooth+beam+hold) : 8.0847
  tabular  (pf1/2/3 GBM OOF)     : 7.7959

=== ブレンド掃引  tvt = w*heur + (1-w)*gbm ===
  w_heur=0.00 (heur:gbm≈0:10)  rmse=7.7959
  w_heur=0.05 (heur:gbm≈0:10)  rmse=7.7350
  w_heur=0.10 (heur:gbm≈1:9)  rmse=7.6816
  w_heur=0.15 (heur:gbm≈2:8)  rmse=7.6360
  w_heur=0.20 (heur:gbm≈2:8)  rmse=7.5983
  w_heur=0.25 (heur:gbm≈2:8)  rmse=7.5685
  w_heur=0.30 (heur:gbm≈3:7)  rmse=7.5468
  w_heur=0.35 (heur:gbm≈4:6)  rmse=7.5333
  w_heur=0.40 (heur:gbm≈4:6)  rmse=7.5280
  w_heur=0.45 (heur:gbm≈4:6)  rmse=7.5309
  w_heur=0.50 (heur:gbm≈5:5)  rmse=7.5420
  w_heur=0.55 (heur:gbm≈6:4)  rmse=7.5613
  w_heur=0.60 (heur:gbm≈6:4)  rmse=7.5887
  w_heur=0.65 (heur:gbm≈6:4)  rmse=7.6241
  w_heur=0.70 (heur:gbm≈7:3)  rmse=7.6674
  w_heur=0.75 (heur:gbm≈8:2)  rmse=7.7185
  w_heur=0.80 (heur:gbm≈8:2)  rmse=7.7772
  w_heur=0.85 (heur:gbm≈8:2)  rmse=7.8434
  w_heur=0.90 (heur:gbm≈9:1